## Quiz 05 - Parallel Computing, Reproducibility, and Containers

### Instructions

This quiz is based on the material covered in lectures 20 to 24. You may use any resources available to you, including the lecture notes and the internet.

All the data required for this quiz can be found in the `data` folder within this repository. If you need to recreate the datasets, you can do so by running the Python script included in the `script-data-generation` folder.

**Important:** Please start by completing Question 01 to set up the correct Python environment before proceeding with the other questions.

Answer the questions in the cells below.
When you are done, please submit your work as an `.html` file on Canvas, or share a link to a GitHub repository with your answers.

### **Question 01: Setting up the Python Environment**

Before proceeding with the rest of the quiz, it is important to set up a Python environment with specific package versions to ensure compatibility and reproducibility. This quiz requires **Python 3.12** and the following packages with exact versions:
- `dask=2026.3.0`
- `duckdb=1.5.1`
- `ipykernel=7.2.0`
- `joblib=1.5.3`
- `numpy=2.4.3`
- `pandas=3.0.2`
- `pyarrow=23.0.1`

You can use tools like `conda`, `pipenv`, or `uv` to manage your environment. If you use conda (recommended), please make sure you **create the environment and install all packages in the same command**. Also include `-c conda-forge` in your command. Make sure to change your current environment to the new environment after creation.

Write the terminal commands in the code cell below:

In [ ]:
# Please write your bash commands here. You can run them using the `!` operator or the `%%bash` magic.
conda create -n quizenv -c conda-forge python=3.12 dask=2026.3.0 duckdb=1.5.1 ipykernel=7.2.0 joblib=1.5.3 numpy=2.4.3 pandas=3.0.2 pyarrow=23.0.1
conda activate quizenv


### Question 02: Understanding the `map` Function and Parallelism

The built-in Python `map()` function applies a function to each element sequentially. Using `joblib`, rewrite the following serial code to run in parallel using **all available cores**. Compare the results to verify correctness.

```python
import numpy as np

def cube_root(x):
    return x ** (1/3)

numbers = np.arange(1, 500001)

# Serial version using map
serial_result = list(map(cube_root, numbers))
print("First 5 serial results:", serial_result[:5])
```

In [4]:
# Please write your answer here.
import numpy as np
from joblib import Parallel, delayed

def cube_root(x):
    return x ** (1/3)

numbers = np.arange(1, 500001)

serial_result = list(map(cube_root, numbers))
print("First 5 serial results:", serial_result[:5])

parallel_result = Parallel(n_jobs=-1)(
    delayed(cube_root)(x) for x in numbers
)

print("First 5 parallel results:", parallel_result[:5])
print("Results match:", np.allclose(serial_result, parallel_result))

First 5 serial results: [np.float64(1.0), np.float64(1.2599210498948732), np.float64(1.4422495703074083), np.float64(1.5874010519681994), np.float64(1.7099759466766968)]
First 5 parallel results: [np.float64(1.0), np.float64(1.2599210498948732), np.float64(1.4422495703074083), np.float64(1.5874010519681994), np.float64(1.7099759466766968)]
Results match: True


### Question 03: Measuring Parallel Speedup

Create a function called `simulate_computation` that generates 100,000 random numbers and calculates their variance. Using `%timeit`, measure and compare the execution time of:

1. Running the function **4 times sequentially** in a list comprehension (`[simulate_computation() for _ in range(4)]`)
2. Running the function **4 times in parallel** using `joblib` with 4 workers

Print and compare both timing results.

In [2]:
# Please write your answer here.
import numpy as np
from joblib import Parallel, delayed

def simulate_computation():
    data = np.random.rand(100_000)
    return np.var(data)

seq_timing = %timeit -o [simulate_computation() for _ in range(4)]

par_timing = %timeit -o Parallel(n_jobs=4)(delayed(simulate_computation)() for _ in range(4))

print(f"\nAverage sequential time: {seq_timing.average:.4f} s")
print(f"Average parallel time:   {par_timing.average:.4f} s")
print(f"Speedup (seq / par):     {seq_timing.average / par_timing.average:.2f}x")

1.84 ms ± 101 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
12.3 ms ± 976 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)

Average sequential time: 0.0018 s
Average parallel time:   0.0123 s
Speedup (seq / par):     0.15x


### Question 04: Dask Array with Custom Chunk Sizes

Create a Dask array of shape (5000, 2000) filled with random integers between 1 and 100. Use chunks of size (500, 500). Then:

1. Compute the sum of each row
2. Calculate the mean and standard deviation of the entire array
3. Print all three results

In [3]:
# Please write your answer here.
import dask.array as da

x = da.random.randint(1, 101, size=(5000, 2000), chunks=(500, 500))

row_sum = x.sum(axis=1)
mean = x.mean()
std = x.std()

print(row_sum.compute())
print(mean.compute())
print(std.compute())

[100046  99970 101852 ... 101343  98267 100947]
50.4950291
28.863606688876448


### Question 05: Optimising Chunk Size

The chunk size significantly affects Dask performance. Create a Dask array with 100,000 random numbers and test three different chunk sizes: 1,000 (many small chunks), 10,000 (medium chunks), and 50,000 (few large chunks).

For each configuration, measure the time to compute `mean(sin(x) + cos(x))`. Which chunk size performed best? Explain why in a comment.

In [ ]:
# Please write your answer here.
import dask.array as da

x1 = da.random.random(100_000, chunks=1_000)
expr1 = da.sin(x1) + da.cos(x1)
t1 = %timeit -o expr1.mean().compute()

x2 = da.random.random(100_000, chunks=10_000)
expr2 = da.sin(x2) + da.cos(x2)
t2 = %timeit -o expr2.mean().compute()

x3 = da.random.random(100_000, chunks=50_000)
expr3 = da.sin(x3) + da.cos(x3)
t3 = %timeit -o expr3.mean().compute()

print("\nAverage times (s):")
print("chunks=1,000 :", t1.average)
print("chunks=10,000:", t2.average)
print("chunks=50,000:", t3.average)

# Best chunk size: 50,000
# This is because it has the smallest average execution time.

34.4 ms ± 1.25 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
5.19 ms ± 181 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2.27 ms ± 19.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

Average times (s):
chunks=1,000 : 0.03444902380002272
chunks=10,000: 0.005194247620009784
chunks=50,000: 0.00227206714427926


### Question 06: Reading Parquet Files with Column Selection

The `data` folder contains Parquet files for multiple countries. Using Dask, read **all Parquet files at once** (`data/*.parquet`), but load only the `year` and `population` columns.

Calculate the total world population for each year across all countries and display the results sorted by year.

In [1]:
# Please write your answer here.
import dask.dataframe as dd

df = dd.read_parquet(
    "data/*.parquet",
    columns=["year", "population"],
)

world_pop = df.groupby("year")["population"].sum().compute().sort_index()

print(world_pop)

year
1945     566994202
1946     596804909
1947     606569895
1948     637303888
1949     644613118
           ...    
2019    1893887207
2020    1959057915
2021    1928046753
2022    1985837056
2023    1980706538
Name: population, Length: 79, dtype: int64


### Question 07: DuckDB with Multiple Conditions

Load the `data.csv` file into a Pandas DataFrame. Using DuckDB, write a SQL query that:

1. Selects countries where `gdp_per_capita` was between 10000 and 50000
2. Filters for years between 2000 and 2020
3. Orders results by `gdp_per_capita` in descending order
4. Limits to the top 2 results

Execute the query and display the results.

In [6]:
# Please write your answer here.
import pandas as pd
import duckdb

df = pd.read_csv("data/data.csv")

result = duckdb.query("""
    SELECT country, year, gdp_per_capita
    FROM df
    WHERE gdp_per_capita BETWEEN 10000 AND 50000
      AND year BETWEEN 2000 AND 2020
    ORDER BY gdp_per_capita DESC
    LIMIT 2
""").to_df()

print(result)

  country  year  gdp_per_capita
0     USA  2002    48942.492140
1     USA  2003    47607.365171


### Question 08: DuckDB with Aggregation

Using the same `data.csv` file, write a SQL query with DuckDB that calculates:

1. The average GDP per capita for each country
2. The minimum and maximum years in the dataset for each country

Group by country and display all results.

In [7]:
# Please write your answer here.
import pandas as pd
import duckdb

df = pd.read_csv("data/data.csv")

result = duckdb.query("""
    SELECT
        country,
        AVG(gdp_per_capita) AS avg_gdp_per_capita,
        MIN(year) AS min_year,
        MAX(year) AS max_year
    FROM df
    GROUP BY country
    ORDER BY country
""").to_df()

print(result)

  country  avg_gdp_per_capita  min_year  max_year
0  Brazil         5496.292031      1945      2023
1   India         1251.704443      1945      2023
2      UK        27496.851363      1945      2023
3     USA        40189.822290      1945      2023


### Question 09: Generating `requirements.txt` and `environment.yml` Files

Write the commands to:

1. Export your current environment's packages to a `requirements.txt` and an `environment.yml` file
2. Show how someone else would install these exact dependencies in these two cases

Explain each step with comments. It is not necessary to run the code.

In [ ]:
# Please write your answer here.
# Export the current conda environment to environment.yml
conda env export -n quizenv > environment.yml

# Export the current environment's Python packages to requirements.txt
pip freeze > requirements.txt

# Install dependencies from requirements.txt using pip
pip install -r requirements.txt

# Install dependencies from environment.yml using conda
conda env create -f environment.yml

### Question 10: Troubleshooting a Broken Dockerfile

The following Dockerfile has several errors. Identify and fix 5 issues, then explain what was wrong with each line:

```dockerfile
# Broken Dockerfile - Fix the errors
from ubuntu

RUN apt install python3 python3-pip
RUN pip install numpy pandas

COPY . .
EXPOSE 8888
RUN ["python3", "app.py"]
```

Write the corrected Dockerfile and list each error with its fix.

Corrected Dockerfile:

```dockerfile
FROM ubuntu:24.04

RUN apt-get update && \
    apt-get install -y python3 python3-pip

WORKDIR /app
COPY . /app

RUN python3 -m pip install --no-cache-dir numpy pandas

EXPOSE 8888

CMD ["python3", "app.py"]
```

1. from ubuntu → FROM ubuntu:24.04
    Docker instructions should use FROM and a pinned version

2. RUN apt install python3 python3-pip
    It is missing apt-get update and -y

3. Missing WORKDIR before COPY
    You need to specify file paths to the directory you want, this only directs it to /

4. RUN pip install numpy pandas
    Should use python3 -m pip to run the correct version.

5. RUN ["python3", "app.py"]
    Should be CMD and not RUN to run container time

### Question 11 - Writing a Dockerfile to Install Software on a Base Image

Create a Dockerfile that starts from an Ubuntu image and installs the following software:

- Git version 1:2.43.0-1ubuntu7.3
- SQLite version 3.45.1-1ubuntu2

Ensure that you specify the exact versions of the packages by checking their versions after installation. Include commands to clean up the package manager cache after installation to reduce the image size.

```dockerfile
FROM ubuntu:24.04

RUN apt-get update && \
    apt-get install -y \
    git=1:2.43.0-1ubuntu7.3 \
    sqlite3=3.45.1-1ubuntu2 && \
    apt-get clean && \
    rm -rf /var/lib/apt/lists/*

RUN git --version && sqlite3 --version
```

#### Please write your anwer here. You can use ```dockerfile to format your code

### Question 12: Dockerfile for a Jupyter Data Science Environment

Create a Dockerfile starting from Ubuntu that:

1. Installs Python 3.12 and pip
2. Installs `jupyterlab`, `numpy`, `pandas`, `matplotlib`, and `scikit-learn` with specific versions of your choice
3. Sets the working directory to `/home/analyst/notebooks`
4. Exposes port 8888
5. Starts JupyterLab with `--no-browser` and `--ip=0.0.0.0`

Clean up apt cache to reduce image size.

#### Please write your answer here. You can use ```dockerfile to format your code

```dockerfile
FROM ubuntu:24.04

RUN apt-get update && \
    apt-get install -y python3.12 python3.12-venv python3-pip && \
    apt-get clean && \
    rm -rf /var/lib/apt/lists/*

RUN python3.12 -m pip install --no-cache-dir \
    jupyterlab==4.1.5 \
    numpy==1.26.4 \
    pandas==2.2.1 \
    matplotlib==3.8.3 \
    scikit-learn==1.4.1

WORKDIR /home/analyst/notebooks

EXPOSE 8888

CMD ["jupyter-lab", "--no-browser", "--ip=0.0.0.0", "--port=8888", "--allow-root"]
```